# MixesDB search → tracklists

Enter a search word (e.g. `hawtin`). This notebook:
1. Searches MixesDB for mixes whose **title** contains the word (via `mixesdb_scraper.py`)
2. Fetches the tracklist for each matching mix (via `mixesdb_tracklist.py`)
3. Prints the tracklist for every mix that has one, and skips those without
4. Optionally saves everything to a single CSV

A small delay is added between requests (paginated search/category batches, and each tracklist fetch) to stay polite to the server and avoid getting rate-limited/blocked.

In [ ]:
import sys, os, time
sys.path.insert(0, os.getcwd())

from mixesdb_scraper import search
from mixesdb_tracklist import get_tracklist
import pandas as pd

## 1. Enter the search word

In [ ]:
query = input("Search word: ").strip()
max_mixes = 20  # how many matching mixes to fetch tracklists for; raise if you want more

print(f"Searching MixesDB for '{query}'...")

## 2. Run the search (title must contain the word)

In [ ]:
results = search(query, limit=max_mixes, strict=True)

print(f"Found {len(results)} mix(es) with '{query}' in the title.\n")
pd.DataFrame(results)[["title", "url"]] if results else print("No matching mixes found.")

## 3. Fetch and display the tracklist for each mix

In [ ]:
REQUEST_DELAY = 0.5  # seconds between page fetches, be polite to the server / avoid getting blocked

all_tracks = []  # flat rows for the CSV export: mix_title, mix_url, part, position, timestamp, artist, title, raw

for i, r in enumerate(results):
    if i > 0:
        time.sleep(REQUEST_DELAY)

    print(f"\n{'='*80}\n{r['title']}\n{r['url']}\n{'='*80}")

    try:
        tracks = get_tracklist(r["url"])
    except Exception as e:
        print(f"  Could not fetch tracklist: {e}")
        continue

    if not tracks:
        print("  No tracklist found on this page.")
        continue

    current_part = None
    for t in tracks:
        if t["part"] != current_part:
            current_part = t["part"]
            if current_part:
                print(f"\n-- {current_part} --")
        ts = f"[{t['timestamp']}] " if t["timestamp"] else ""
        line = f"{t['artist']} - {t['title']}" if t["artist"] else t["raw"]
        print(f"{t['position']:>3}. {ts}{line}")

        all_tracks.append({
            "mix_title": r["title"],
            "mix_url": r["url"],
            **t,
        })

print(f"\n\nDone. Collected {len(all_tracks)} tracks across {len(results)} mix(es).")

## 4. (optional) Save all tracks to CSV

In [ ]:
if all_tracks:
    out_path = f"{query.replace(' ', '_')}_tracklists.csv"
    df = pd.DataFrame(all_tracks)
    df.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"Saved {len(df)} rows to {out_path}")
else:
    print("Nothing to save — no tracklists were found.")